<a href="https://colab.research.google.com/github/Omdevkar67/Data_Science_Lab/blob/main/Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Breadth-First Search (BFS)

BFS is an algorithm for traversing or searching tree or graph data structures. It starts at the tree root (or some arbitrary node of a graph, sometimes referred to as a 'search key') and explores all of the neighbor nodes at the present depth prior to moving on to the nodes at the next depth level.

It uses a queue data structure to keep track of the nodes to visit.

In [7]:
from collections import deque

def bfs(graph, start_node):
    visited = set()
    queue = deque([start_node])
    bfs_path = []

    visited.add(start_node)

    while queue:
        current_node = queue.popleft()
        bfs_path.append(current_node)

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
    return bfs_path

# Example Graph (Adjacency List)
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

print("BFS Traversal starting from 'A':", bfs(graph, 'A'))

BFS Traversal starting from 'A': ['A', 'B', 'C', 'D', 'E', 'F']


### Depth-First Search (DFS)

DFS is an algorithm for traversing or searching tree or graph data structures. The algorithm starts at the root node (selecting some arbitrary node as the root node in the case of a graph) and explores as far as possible along each branch before backtracking.

It typically uses a stack data structure (implicitly via recursion or explicitly with a list) to keep track of the nodes to visit.

In [8]:
def dfs(graph, start_node):
    visited = set()
    dfs_path = []

    def dfs_recursive(node):
        visited.add(node)
        dfs_path.append(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                dfs_recursive(neighbor)

    dfs_recursive(start_node)
    return dfs_path

# Using the same example graph
print("DFS Traversal starting from 'A':", dfs(graph, 'A'))

DFS Traversal starting from 'A': ['A', 'B', 'D', 'E', 'F', 'C']


### A* Search Algorithm

A* (pronounced "A-star") is a graph traversal and pathfinding algorithm, which is often used in computer science for finding the shortest path between two nodes in a graph. It is a 'best-first search' algorithm, meaning it explores the most promising paths first.

A* uses a heuristic function to estimate the cost from the current node to the goal node, adding this to the actual cost from the start node to the current node. This combined cost is used to determine the order in which the algorithm explores paths.

It requires:
- **`g(n)`**: The cost from the start node to node `n`.
- **`h(n)`**: The heuristic estimate of the cost from node `n` to the goal node.
- **`f(n) = g(n) + h(n)`**: The total estimated cost of the path through `n` to the goal.

In [9]:
import heapq

def a_star(graph, start, goal, heuristic):
    # The set of discovered nodes that may need to be (re-)expanded.
    # Initially, only the start node is known.
    # This is a min-heap, storing (f_score, node)
    open_set = [(heuristic[start], start)]

    # For node n, came_from[n] is the node immediately preceding it on the cheapest path from start
    # to n currently known.
    came_from = {}

    # For node n, g_score[n] is the cost of the cheapest path from start to n currently known.
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0

    # For node n, f_score[n] = g_score[n] + h[n]. f_score[n] represents our current best guess as to
    # how cheap a path could be from start to finish if it goes through n.
    f_score = {node: float('inf') for node in graph}
    f_score[start] = heuristic[start]

    while open_set:
        # current is the node in open_set having the lowest f_score value
        current_f, current = heapq.heappop(open_set)

        if current == goal:
            return reconstruct_path(came_from, current)

        for neighbor, weight in graph[current].items():
            # d(current, neighbor) is the weight of the edge from current to neighbor
            tentative_g_score = g_score[current] + weight

            if tentative_g_score < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = tentative_g_score + heuristic[neighbor]
                heapq.heappush(open_set, (f_score[neighbor], neighbor))

    # Open set is empty but goal was never reached
    return None

def reconstruct_path(came_from, current):
    total_path = [current]
    while current in came_from:
        current = came_from[current]
        total_path.append(current)
    return total_path[::-1] # Reverse path to get it from start to goal

# Example Graph (Adjacency List with weights)
# Format: {node: {neighbor: weight, ...}, ...}
graph_astar = {
    'A': {'B': 1, 'C': 4},
    'B': {'A': 1, 'D': 2, 'E': 5},
    'C': {'A': 4, 'F': 2},
    'D': {'B': 2},
    'E': {'B': 5, 'F': 1},
    'F': {'C': 2, 'E': 1}
}

# Heuristic values (straight-line distance to goal 'F')
heuristic_astar = {
    'A': 6,
    'B': 4,
    'C': 2,
    'D': 7,
    'E': 1,
    'F': 0
}

start_node_astar = 'A'
goal_node_astar = 'F'

path = a_star(graph_astar, start_node_astar, goal_node_astar, heuristic_astar)

if path:
    print(f"A* Path from {start_node_astar} to {goal_node_astar}: {path}")
else:
    print(f"No path found from {start_node_astar} to {goal_node_astar}")

A* Path from A to F: ['A', 'C', 'F']


### Minimax Algorithm

Minimax is a decision-making algorithm commonly used in artificial intelligence for two-player zero-sum games (where one player's gain is another's loss). It works by building a game tree that represents all possible future moves and their outcomes. The algorithm then attempts to minimize the maximum possible loss for a worst-case scenario. It assumes the opponent plays optimally.

**Key Concepts:**
- **Maximizing Player:** Aims to maximize their score.
- **Minimizing Player:** Aims to minimize the opponent's (and thus maximize their own) score.
- **Terminal State:** A game state where the game has ended (win, loss, or draw).
- **Evaluation Function:** Assigns a numerical score to a terminal (or non-terminal) game state.

In [10]:
import math

def minimax(state, depth, maximizing_player):
    # Base cases: Check if the game is over or depth limit is reached
    if is_terminal(state):
        return evaluate(state)

    if depth == 0:
        return evaluate(state) # Or a heuristic evaluation for non-terminal states

    if maximizing_player:
        max_eval = -math.inf
        for move in get_possible_moves(state):
            new_state = make_move(state, move)
            eval = minimax(new_state, depth - 1, False)
            max_eval = max(max_eval, eval)
        return max_eval
    else:
        min_eval = math.inf
        for move in get_possible_moves(state):
            new_state = make_move(state, move)
            eval = minimax(new_state, depth - 1, True)
            min_eval = min(min_eval, eval)
        return min_eval

# --- Placeholder functions for a specific game implementation ---
# These functions need to be defined based on the game you are playing.

def is_terminal(state):
    """Checks if the given game state is a terminal state (game over)."""
    # Example: return True if a player has won or it's a draw
    return False

def evaluate(state):
    """Evaluates the score of a terminal (or non-terminal) game state.
    Positive values for maximizing player, negative for minimizing player."""
    # Example: return 1 if maximizing player wins, -1 if minimizing player wins, 0 for draw
    return 0

def get_possible_moves(state):
    """Returns a list of all legal moves from the current state."""
    # Example: return ['move1', 'move2']
    return []

def make_move(state, move):
    """Applies a move to the current state and returns the new state."""
    # Example: return state_after_move
    return state

# Example Usage (conceptual, as game-specific functions are not defined):
# initial_game_state = 'some_representation_of_the_board'
# optimal_score = minimax(initial_game_state, depth=3, maximizing_player=True)
# print("Optimal score for maximizing player:", optimal_score)

print("Minimax function defined. Remember to implement `is_terminal`, `evaluate`, `get_possible_moves`, and `make_move` for your specific game.")


Minimax function defined. Remember to implement `is_terminal`, `evaluate`, `get_possible_moves`, and `make_move` for your specific game.
